In [15]:
import pandas as pd

def get_clearing_price(my_p, my_v, bids, asks):
    # Potential clearing prices are all prices present in the book
    prices = sorted(list(set(list(bids.keys()) + list(asks.keys()) + [my_p])))
    best_cp = 0
    max_vol = -1
    
    for p in prices:
        # Demand: Others >= p, plus You if your bid >= p
        demand = sum(v for price, v in bids.items() if price >= p)
        if my_p >= p:
            demand += my_v
            
        # Supply: Asks <= p
        supply = sum(v for price, v in asks.items() if price <= p)
        
        traded_vol = min(demand, supply)
        
        if traded_vol > max_vol:
            max_vol = traded_vol
            best_cp = p
        elif traded_vol == max_vol:
            best_cp = max(best_cp, p) # Rule: break ties with higher price
            
    return best_cp

def calculate_exact_fill(my_p, my_v, cp, bids, asks):
    if my_p < cp:
        return 0 # You didn't meet the clearing price
    
    # Total supply available at or below the clearing price
    total_supply = sum(v for p, v in asks.items() if p <= cp)
    
    # Volume with priority over you:
    # 1. Bids strictly higher than yours (Price Priority)
    # 2. Bids equal to yours (Time Priority - you are last)
    vol_ahead = sum(v for p, v in bids.items() if p > my_p)
    vol_at_my_level = sum(v for p, v in bids.items() if p == my_p)
    
    priority_volume = vol_ahead + vol_at_my_level
    
    # Your fill is what's left of the total supply after priority volume is taken
    fill = max(0, total_supply - priority_volume)
    return min(my_v, fill)

# Example Optimization Loop
def find_best_bid(bids, asks, buyback, fee=0):
    results = []
    # Try prices and a range of volumes
    for p_test in range(min(bids), buyback + 2):
        for v_test in range(0, 100000, 1): # Adjust range/step based on book
            cp = get_clearing_price(p_test, v_test, bids, asks)
            fill = calculate_exact_fill(p_test, v_test, cp, bids, asks)
            profit = fill * (buyback - cp - fee)
            results.append((p_test, v_test, cp, fill, profit))
            
    return pd.DataFrame(results, columns=['BidP', 'BidV', 'CP', 'Fill', 'Profit']).sort_values('Profit', ascending=False)

# INPUT DATA (Replace with your actual UI numbers)
flax_bids = {30: 30000, 29: 5000, 28: 12000, 27: 28000}
flax_asks = {28: 40000, 31: 20000, 32: 20000, 33: 30000}

mushroom_bids = {20: 43000, 19: 17000, 18: 6000, 17: 5000, 16: 10000, 15: 5000, 14: 10000, 13: 7000} 
mushroom_asks = {12: 20000, 13: 25000, 14: 35000, 15: 6000, 16: 5000, 18: 10000, 19: 12000} 

print("Best Flax Strategy:")
print(find_best_bid(flax_bids, flax_asks, 30).head(1))

print("\nBest Mushroom Strategy:")
print(find_best_bid(mushroom_bids, mushroom_asks, 20, fee=0.10).head(1))

Best Flax Strategy:
        BidP  BidV  CP  Fill  Profit
309999    30  9999  29  9999    9999

Best Mushroom Strategy:
        BidP   BidV  CP   Fill   Profit
719999    20  19999  16  19999  77996.1


In [13]:
a = 9999 + 77996
print(a)

87995
